Tugas5_2505060019_Syifa Rahma Rasendriya

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 17:16:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [11]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("TugasMandiri5") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

df_target_cabang = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.show(5)
df_target_cabang.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



[Stage 28:======================================>                   (2 + 1) / 3]

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [12]:
#A. join dan Perbandingan Target

from pyspark.sql.functions import sum as spark_sum
df_total = df_transaksi.groupBy("kota").agg(spark_sum("pendapatan").alias("total_pendapatan"))
presentase_pencapaian = df_total.join(df_target_cabang, on="kota", how="inner")
presentase_pencapaian = presentase_pencapaian.withColumn("pencapaian_persen", col("total_pendapatan") / col("target_bulanan") * 100)
presentase_pencapaian = presentase_pencapaian.orderBy(col("pencapaian_persen").desc())
presentase_pencapaian.show()

[Stage 32:===========================================>              (3 + 1) / 4]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [13]:
#B. Window Function - Kategori Terlaris per Kota

from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc
df_kategori = df_transaksi.groupby("kota", "kategori").agg(spark_sum("pendapatan").alias("total_pendapatan"))
window_kategori = Window.partitionBy("kota").orderBy(desc("total_pendapatan"))
peringkat_pendapatan = df_kategori.withColumn("peringkat", row_number().over(window_kategori))
peringkat_pendapatan = peringkat_pendapatan.filter(col("peringkat") == 1)
peringkat_pendapatan.show()

+----------+--------------------+----------------+---------+
|      kota|            kategori|total_pendapatan|peringkat|
+----------+--------------------+----------------+---------+
|  Magelang|Kesehatan & Kecan...|         7275000|        1|
| Purworejo|Kesehatan & Kecan...|        10075000|        1|
|  Semarang|        Rumah Tangga|        11125000|        1|
|      Solo|Kesehatan & Kecan...|         8425000|        1|
|Yogyakarta|             Fashion|        13325000|        1|
+----------+--------------------+----------------+---------+



In [14]:
#C. Spark SQL

df_transaksi.createOrReplaceTempView("transaksi")
df_target_cabang.createOrReplaceTempView("target")

data_ringkasan = spark.sql("""
    SELECT
        t.kota,
        tg.pic_cabang,
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    INNER JOIN target tg
        ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

data_ringkasan.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



#D. Kesimpulan

Dari hasil data A dan B, cabang yang kinerjanya paling baik bisa dilihat dari pendaapatan target bulanannya dan juga kategori yang punya 
pendapatan paling tinggi. Purworejo memiliki pencapaian seebesar 152,17%, dengan total pendapatannya Rp45.650.000 dari targetnya 
Rp30.000.000. Purworejo juga yang jumlah transaksinya paling banyak, yaitu 116 transaksi dan kategori yang pendapatannya paling tinggi 
adalah Kesehatan & Kecantikan yang memperoleh sebesar Rp10.075.000. Sedangkan, yang memiliki pennsapaian paling rendah adalah Semarang yaitu
69.41% dengan total pendapatannya Rp38.175.000 dari targetnya Rp55.000.000. JUmlah transaksinya juga sedikit dibanding yang lain, yaitu 93
transaksi, sedangkan kategori paling tinggi  adalah Rumah Tangga yang memperoleh sebesar Rp11.125.000. Dari angka itu, bisa dilihat kalau
setiap cabang itu mempunyai kondisi yang berbeda antara jumlah transaksi, total pendapatan, target, dan kategori yang paling banyak 
menghasilkan pendapatam. Dari data itu juga bisa dilihat kalau cabang yang paling bayak memerlukan penanganan serius dari puhak manajemen
adalah cabang Semarang karena dibandingkan dengan yang lainnya persentase pencapaian target bulanannya paling rendah dibandingkan cabang 
dengan lainnya.